#Introduction
This project builds on *"NetFlow Datasets for Machine Learning-based Network Intrusion Detection Systems"* (Sarhan et al., 2020), which converted four NIDS benchmark datasets into a shared 12-feature NetFlow format. Three of those datasets are used here - NF-UNSW-NB15, NF-BoT-IoT, and NF-ToN-IoT - to compare three algorithms (Random Forest, Extra Trees, XGBoost) across three differently-sourced network environments, finishing with a cross-dataset comparison.

The same 8 behavioral features and the same 3 models are used for every dataset, so the full pipeline is only fully explained once (Part A) - Parts B and C repeat it without re-deriving the reasoning.

In [ ]:
# Cell 1: Import Libraries and Helpers
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# For model building
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    )
from xgboost import XGBClassifier

# For handling imbalance
from imblearn.over_sampling import SMOTE

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline


# **DATASET 1 : NF-UNSW-NB15**

##importing dataset

Upload Kaggle API Key

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# Move the file to the correct directory and set permissions
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download dhoogla/nfunswnb15


In [ ]:
!unzip nfunswnb15.zip


##Collect Data

In [ ]:
df1 = pd.read_parquet('NF-UNSW-NB15.parquet')
df1.shape


In [ ]:
df1.head()



### DESCRIPTION of the Dataset:


| Feature Name | Description |
| :--- | :--- |
| **L4_SRC_PORT** | Source port number (Transport layer) |
| **L4_DST_PORT** | Destination port number (Transport layer) |
| **PROTOCOL** | IP protocol identifier (e.g., TCP=6, UDP=17, ICMP=1) |
| **TCP_FLAGS** | Cumulative TCP flags (SYN, ACK, FIN, RST, etc.) |
| **L7_PROTO** | Layer 7 protocol (Application layer protocol) |
| **IN_BYTES** | Number of incoming bytes |
| **OUT_BYTES** | Number of outgoing bytes |
| **IN_PKTS** | Number of incoming packets |
| **OUT_PKTS** | Number of outgoing packets |
| **FLOW_DURATION_MILLISECONDS** | Flow duration in milliseconds |
| **Label** | Binary label: 0=Benign, 1=Attack |
| **Attack** | Multi-class attack category (Specific attack type) |


### Sampling for tractability
**Sampling for tractability**: the full dataset is 1.48M rows. Training three ensemble models (with SMOTE and 5-fold CV) on the full size is unnecessarily slow for what's fundamentally a feature/algorithm comparison exercise, not a production model - taking a stratified sample (preserving the original Label ratio) keeps training fast while still giving each model a large, representative sample to learn from.

In [ ]:
SAMPLE_SIZE = 150_000

df1, _ = train_test_split(df1, train_size=SAMPLE_SIZE, stratify=df1['Label'], random_state=42)
df1 = df1.reset_index(drop=True)

df1.shape


##Exploratory Data Analysis (EDA)


In [ ]:
df1.shape


In [ ]:
df1.info()


In [ ]:
# check for missing values
df1.isna().sum()


No missing values

In [ ]:
# check for duplicate rows
df1.duplicated().sum()


no duplicate rows

###TARGET VARIABLE ANALYSIS - BINARY CLASSIFICATION

In [ ]:
df1['Label'].value_counts()


All 12 columns of this NetFlow build are available: `L4_SRC_PORT`, `L4_DST_PORT`, `PROTOCOL`, `L7_PROTO`, `IN_BYTES`, `OUT_BYTES`, `IN_PKTS`, `OUT_PKTS`, `TCP_FLAGS`, `FLOW_DURATION_MILLISECONDS`, plus `Label` (binary) and `Attack` (subcategory, leakage for the binary task since it's only populated for attack rows). This build doesn't include the raw `IPV4_SRC_ADDR`/`IPV4_DST_ADDR` fields at all - they're already stripped out upstream - so only the two port columns need to be dropped for the same identifier-leakage reason. `PROTOCOL` and `L7_PROTO` are also already numerically encoded in this build (int8/float32), not raw strings, so no additional categorical encoding is needed.

## Feature Selection & Engineering

**Selected (8 behavioral features):** `PROTOCOL`, `TCP_FLAGS`, `L7_PROTO`, `IN_BYTES`, `OUT_BYTES`, `IN_PKTS`, `OUT_PKTS`, `FLOW_DURATION_MILLISECONDS`

**Dropped:** `L4_SRC_PORT`, `L4_DST_PORT`, `Attack`

**Justification:** source/destination ports are host/session identifiers tied to specific machines and connections in this capture - a model trained on them memorizes "this port is the attacker's" instead of learning general attack *behavior*, and would fail on any new network (the paper drops ports and IPs for the same reason, Section 4 - this build has already stripped the IP columns upstream). `Attack` is dropped because it's only populated for attack rows - leakage for the binary target. The 8 behavioral features describe how a flow behaved (volume, duration, protocol/flag patterns), which is what should generalize.

In [ ]:
selected_features = ['PROTOCOL', 'TCP_FLAGS', 'L7_PROTO', 'IN_BYTES', 'OUT_BYTES',
                     'IN_PKTS', 'OUT_PKTS', 'FLOW_DURATION_MILLISECONDS']
target_col = 'Label'

X1 = df1[selected_features].copy()
y1 = df1[target_col].astype(int)


**Verifying the selection** - checking that the 8 selected features actually separate benign from attack traffic before committing to them.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 22))
axes = axes.flatten()

for ax, feature in zip(axes, selected_features):
    benign_data = X1[y1 == 0][feature]
    attack_data = X1[y1 == 1][feature]

    bp = ax.boxplot([benign_data, attack_data], labels=['Benign', 'Attack'], medianprops={'color': "#f5f500", 'linewidth': 2}, patch_artist=True)

    colors = ['#2ecc71', "#e73c3c"] # Soft green -> Benign, Soft red ->Attack
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title(f"Distribution of {feature}", fontweight='bold', fontsize=14, pad=10)
    ax.grid(True, linestyle='--', alpha=0.5, axis='y')

    if feature in ['IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS']:
        ax.set_yscale('log')

plt.tight_layout()
plt.subplots_adjust(hspace=0.4, wspace=0.3)
plt.show()


The selected features show clear class-separating behavior (visible spread between Benign/Attack boxes, especially in byte/packet counts on the log scale) - confirms they're worth keeping.

## Data Preprocessing

 * no missing values or duplicate rows to remove (confirmed in EDA above)


 * PROTOCOL and L7_PROTO are already numerically encoded in this build - no categorical encoding needed

In [ ]:
X1.describe().round(2)


## Split Dataset (Training & Testing) & Handle Imbalance

**Note on scaling:** `StandardScaler` is retained for pipeline consistency. Random Forest, Decision Tree, and XGBoost are tree-based models and generally do not require feature scaling.


In [ ]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, y1, test_size=0.2, random_state=42, stratify=y1)

scaler1 = StandardScaler()
X_train1 = scaler1.fit_transform(X_train1)
X_test1 = scaler1.transform(X_test1)


###TARGET VARIABLE ANALYSIS

In [ ]:
label_counts = df1['Label'].value_counts()
label_percentages = df1['Label'].value_counts(normalize=True) * 100

print(f" Label Distribution:")
print(f"   Benign  (0): {label_counts[0]:,} ({label_percentages[0]:.2f}%)")
print(f"   Attack  (1): {label_counts[1]:,} ({label_percentages[1]:.2f}%)")


visualize

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Pie chart
colors = ['#4CAF50', '#FF5722']
ax.pie([label_counts[0], label_counts[1]],
            labels=['Benign', 'Attack'],
            autopct='%1.2f%%',
            colors=colors,
            explode=(0.05, 0.1),
            shadow=True,
            startangle=90)
ax.set_title('Class Distribution - NF-UNSW-NB15', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


NF-UNSW-NB15 is benign-heavy (~95.5% benign / 4.5% attack) - **applying `SMOTE` to the `training set` only**.

### Conclusion -> `high class imbalance`


### Applying Smote

In [ ]:
smote1 = SMOTE(random_state=42)
X_train1, y_train1 = smote1.fit_resample(X_train1, y_train1)
y_train1.value_counts(normalize=True).round(3)


## Select ML Algorithm
Three tree-ensemble methods - Random Forest (bagging ensemble), Decision Tree (interpretable baseline), and XGBoost (boosting ensemble) - chosen over SVM/KNN/Naive Bayes, which don't scale well to this row count and aren't standard for NIDS work.


In [ ]:
def get_models():
    """Return fresh model instances for each dataset."""
    return {
        'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15, min_samples_split=2,
                                                random_state=42, n_jobs=-1, class_weight=None),
        'Decision Tree': DecisionTreeClassifier(max_depth=15, random_state=42),
        'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                 random_state=42, n_jobs=-1, eval_metric='logloss')
    }

models = get_models()


#### Global Hyperparameter Tuning Configuration and Helper Function

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for Random Forest tuning (same for all datasets)
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5]
}

def tune_and_evaluate_random_forest(X_train, y_train, X_test, y_test, results_dict, predictions_dict, y_probs_dict, dataset_name, param_grid_rf):
    """Performs GridSearchCV for Random Forest, evaluates, and updates results_dict."""
    print(f"\n--- Starting Random Forest Hyperparameter Tuning for {dataset_name} ---")

    # Initialize GridSearchCV
    grid_search_rf = GridSearchCV(
        estimator=RandomForestClassifier(random_state=42, n_jobs=-1, class_weight=None),
        param_grid=param_grid_rf,
        scoring='f1_macro',
        cv=3, # Using a smaller CV for quicker demonstration
        n_jobs=-1,
        verbose=1
    )

    # Fit GridSearchCV
    grid_search_rf.fit(X_train, y_train)

    # Print best parameters and best score
    print(f"Best parameters for Random Forest on {dataset_name}: {grid_search_rf.best_params_}")
    print(f"Best F1-Score for Random Forest on {dataset_name}: {grid_search_rf.best_score_:.4f}")

    best_rf_model_tuned = grid_search_rf.best_estimator_

    # Re-train the best Random Forest model on the full training data (optional, GridSearchCV already does this)
    # best_rf_model_tuned.fit(X_train, y_train) # Not strictly necessary if best_estimator_ is ready

    # Predict and evaluate
    y_pred_tuned_rf = best_rf_model_tuned.predict(X_test)
    y_prob_tuned_rf = best_rf_model_tuned.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred_tuned_rf)
    f1 = f1_score(y_test, y_pred_tuned_rf)
    auc = roc_auc_score(y_test, y_prob_tuned_rf)
    precision = precision_score(y_test, y_pred_tuned_rf, zero_division=0)
    recall = recall_score(y_test, y_pred_tuned_rf, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_tuned_rf, labels=[0, 1]).ravel()
    dr = tp / max((tp + fn), 1) # Detection Rate (DR)
    far = fp / max((fp + tn), 1) # False Alarm Rate (FAR).

    cv_scores = cross_val_score(best_rf_model_tuned, X_train, y_train, cv=5, scoring='f1_macro')

    # Update results_dict, predictions_dict, y_probs_dict with tuned Random Forest results
    results_dict['Random Forest'] = {
        'Accuracy': accuracy,
        'F1-Score': f1,
        'AUC': auc,
        'Precision': precision,
        'Recall': recall,
        'Detection Rate (DR)': dr,
        'False Alarm Rate (FAR)': far,
        'CV Mean F1': cv_scores.mean(),
        'CV Std F1': cv_scores.std(),
    }
    predictions_dict['Random Forest'] = y_pred_tuned_rf
    y_probs_dict['Random Forest'] = y_prob_tuned_rf

    return results_dict, predictions_dict, y_probs_dict, grid_search_rf.best_params_, grid_search_rf.best_score_, best_rf_model_tuned


**NIDS** = Network Intrusion Detection System

---

### Why Other Algorithms Don't Scale Well?

A fundamental rule of thumb when working with network traffic datasets (like `NF-UNSW-NB15` or `NF-UQ-NIDS`):

#### 1. The High Row Count Challenge
Real-world network datasets contain millions of rows because every single connection, ping, or data packet generates a log entry.
* **SVM (Support Vector Machines)** and **KNN (K-Nearest Neighbors)** have high computational complexity relative to the number of rows. If you try to run KNN or SVM on a dataset with 500,000+ network flows, your system will likely run out of memory or take days to complete a single training run.
* **Tree-Ensembles (Random Forest, XGBoost)** handle massive row counts easily due to parallel processing (`n_jobs=-1`) and optimizations like histogram-based splitting.

#### 2. The Complex Feature Relationships
Network attacks are rarely distinct or isolated; they hide inside regular traffic by changing subtle timing patterns or packet sizes.
* **Naive Bayes** assumes all features are completely independent of one another. In NIDS data, this assumption breaks completely because features like `IN_BYTES`, `IN_PKTS`, and `PROTOCOL` are deeply intertwined. Naive Bayes would yield a dangerously high **False Alarm Rate**.
* **Tree-Ensembles** natively capture these non-linear feature interactions (e.g., *“If PROTOCOL is 6 AND L4_DST_PORT is 22 AND IN_PKTS is abnormally high, flag as an attack”*).

By focusing your code on Random Forest, Decision Tree, and XGBoost, you are aligning with current industry-standard practices for processing network telemetry.


## Train & Evaluate Model

In [ ]:
results1 = {}
predictions1 = {}
y_probs1 = {}

for name, model in models.items():

    model.fit(X_train1, y_train1)

    y_pred = model.predict(X_test1)
    y_prob = model.predict_proba(X_test1)[:, 1]

    accuracy = accuracy_score(y_test1, y_pred)
    f1 = f1_score(y_test1, y_pred)
    auc = roc_auc_score(y_test1, y_prob)
    precision = precision_score(y_test1, y_pred, zero_division=0)
    recall = recall_score(y_test1, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test1, y_pred, labels=[0, 1]).ravel()
    dr = tp / max((tp + fn), 1) # Detection Rate (DR)
    far = fp / max((fp + tn), 1) # False Alarm Rate (FAR).

    cv_scores = cross_val_score(model, X_train1, y_train1, cv=5, scoring='f1_macro')

    results1[name] = {
        'Accuracy': accuracy,
        'F1-Score': f1,
        'AUC': auc,
        'Precision': precision,
        'Recall': recall,
        'Detection Rate (DR)': dr,
        'False Alarm Rate (FAR)': far,
        'CV Mean F1': cv_scores.mean(),
        'CV Std F1': cv_scores.std(),
    }
    predictions1[name] = y_pred
    y_probs1[name] = y_prob # Store predicted probabilities

results_df1 = pd.DataFrame(results1).T
results_df1.index.name = 'Model'
results_df1.sort_values(by='F1-Score', ascending=False, inplace=True)
display(results_df1.round(4))


## Hyperparameter Tuning - NF-UNSW-NB15

### Random Forest Hyperparameter Tuning (NF-UNSW-NB15)

In [ ]:
results1, predictions1, y_probs1, best_params_rf1, best_score_rf1, tuned_rf1 = tune_and_evaluate_random_forest(
    X_train1, y_train1, X_test1, y_test1, results1, predictions1, y_probs1, 'NF-UNSW-NB15', param_grid_rf
)


### Re-evaluating Random Forest with Tuned Hyperparameters - NF-UNSW-NB15

In [ ]:
results_df1 = pd.DataFrame(results1).T
results_df1.index.name = 'Model'
results_df1.sort_values(by='F1-Score', ascending=False, inplace=True)
display(results_df1.round(4))


### Summary of Hyperparameter Tuning for NF-UNSW-NB15

The best Random Forest hyperparameters and cross-validation F1 score are displayed dynamically in the tuning-cell output.


## Confusion Matrix Visualization

In [ ]:
fig, axes = plt.subplots(1, len(predictions1), figsize=(6 * len(predictions1), 5))

for idx, (name, y_pred) in enumerate(predictions1.items()):
    cm = confusion_matrix(y_test1, y_pred)

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=axes[idx],
        xticklabels=['Benign', 'Attack'],
        yticklabels=['Benign', 'Attack'],
    )

    axes[idx].set_title(f"{name}\nAccuracy: {results1[name]['Accuracy'] * 100:.2f}%", fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()


## ROC Curve Analysis - NF-UNSW-NB15

In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(10, 8))

for name, y_prob in y_probs1.items():
    fpr, tpr, _ = roc_curve(y_test1, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - NF-UNSW-NB15', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## Feature Importance Analysis

In [ ]:
importance_model_name = 'Random Forest'
importance_model = tuned_rf1

feature_importance1 = pd.Series(importance_model.feature_importances_, index=selected_features).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=feature_importance1.values, y=feature_importance1.index, hue=feature_importance1.index,
            palette='viridis', legend=False)
plt.title('Random Forest Feature Importance - NF-UNSW-NB15')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


# DATASET - 2 : NF-BoT-IoT

##importing dataset

In [ ]:
!kaggle datasets download dhoogla/nfbotiot


In [ ]:
!unzip nfbotiot.zip


##Collect Data

In [ ]:
df2 = pd.read_parquet('NF-BoT-IoT.parquet')
df2.shape


In [ ]:
df2.head()



### DESCRIPTION of the Dataset:


| Feature Name | Description |
| :--- | :--- |
| **L4_SRC_PORT** | Source port number (Transport layer) |
| **L4_DST_PORT** | Destination port number (Transport layer) |
| **PROTOCOL** | IP protocol identifier (e.g., TCP=6, UDP=17, ICMP=1) |
| **TCP_FLAGS** | Cumulative TCP flags (SYN, ACK, FIN, RST, etc.) |
| **L7_PROTO** | Layer 7 protocol (Application layer protocol) |
| **IN_BYTES** | Number of incoming bytes |
| **OUT_BYTES** | Number of outgoing bytes |
| **IN_PKTS** | Number of incoming packets |
| **OUT_PKTS** | Number of outgoing packets |
| **FLOW_DURATION_MILLISECONDS** | Flow duration in milliseconds |
| **Label** | Binary label: 0=Benign, 1=Attack |
| **Attack** | Multi-class attack category (Specific attack type) |


##Exploratory Data Analysis (EDA)

In [ ]:
df2.shape


In [ ]:
df2.info()


In [ ]:
# check for missing values
df2.isna().sum()


No missing values

In [ ]:
# check for duplicate rows
df2.duplicated().sum()


no duplicate rows

###TARGET VARIABLE ANALYSIS - BINARY CLASSIFICATION

In [ ]:
df2['Label'].value_counts()


In [ ]:
label_counts = df2['Label'].value_counts()
label_percentages = df2['Label'].value_counts(normalize=True) * 100

print(f" Label Distribution:")
print(f"   Benign  (0): {label_counts[0]:,} ({label_percentages[0]:.2f}%)")
print(f"   Attack  (1): {label_counts[1]:,} ({label_percentages[1]:.2f}%)")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
# Pie chart
colors = ['#4CAF50', '#FF5722']
ax.pie([label_counts[0], label_counts[1]],
            labels=['Benign', 'Attack'],
            autopct='%1.2f%%',
            colors=colors,
            explode=(0.05, 0.1),
            shadow=True,
            startangle=90)
ax.set_title('Class Distribution - NF-BoT-IoT', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


The NF-BoT-IoT dataset exhibits a significant class imbalance with approximately **97.68% Attack** instances and **2.32% Benign** instances. This is an inverse imbalance compared to the NF-UNSW-NB15 dataset. As confirmed earlier, there are no missing values or duplicate rows in this dataset.

## Feature Selection & Engineering

**Selected (8 behavioral features):** `PROTOCOL`, `TCP_FLAGS`, `L7_PROTO`, `IN_BYTES`, `OUT_BYTES`, `IN_PKTS`, `OUT_PKTS`, `FLOW_DURATION_MILLISECONDS`

**Dropped:** `L4_SRC_PORT`, `L4_DST_PORT`, `Attack`

**Justification:** source/destination ports are host/session identifiers tied to specific machines and connections in this capture - a model trained on them memorizes "this port is the attacker's" instead of learning general attack *behavior*, and would fail on any new network (the paper drops ports and IPs for the same reason, Section 4 - this build has already stripped the IP columns upstream). `Attack` is dropped because it's only populated for attack rows - leakage for the binary target. The 8 behavioral features describe how a flow behaved (volume, duration, protocol/flag patterns), which is what should generalize.

In [ ]:
selected_features = ['PROTOCOL', 'TCP_FLAGS', 'L7_PROTO', 'IN_BYTES', 'OUT_BYTES',
                     'IN_PKTS', 'OUT_PKTS', 'FLOW_DURATION_MILLISECONDS']
target_col = 'Label'

X2 = df2[selected_features].copy()
y2 = df2[target_col].astype(int)


**Verifying the selection** - checking that the 8 selected features actually separate benign from attack traffic before committing to them.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 22))
axes = axes.flatten()

for ax, feature in zip(axes, selected_features):
    benign_data = X2[y2 == 0][feature]
    attack_data = X2[y2 == 1][feature]

    bp = ax.boxplot([benign_data, attack_data], labels=['Benign', 'Attack'], medianprops={'color': "#f5f500", 'linewidth': 2}, patch_artist=True)

    colors = ['#2ecc71', "#e73c3c"] # Soft green -> Benign, Soft red ->Attack
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title(f"Distribution of {feature}", fontweight='bold', fontsize=14, pad=10)
    ax.grid(True, linestyle='--', alpha=0.5, axis='y')

    if feature in ['IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS']:
        ax.set_yscale('log')

plt.tight_layout()
plt.subplots_adjust(hspace=0.4, wspace=0.3)
plt.show()


The selected features show clear class-separating behavior (visible spread between Benign/Attack boxes, especially in byte/packet counts on the log scale) - confirms they're worth keeping.

## Data Preprocessing

* no missing values or duplicate rows to remove (confirmed in EDA above)
* `PROTOCOL` and `L7_PROTO` are already numerically encoded in this build - no categorical encoding needed

## Split Dataset (Training & Testing) & Handle Imbalance

In [ ]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)

scaler2 = StandardScaler()
X_train2 = scaler2.fit_transform(X_train2)
X_test2 = scaler2.transform(X_test2)


### TARGET VARIABLE ANALYSIS

The NF-BoT-IoT dataset exhibits a significant class imbalance with approximately **97.68% Attack** instances and **2.32% Benign** instances. This is an inverse imbalance compared to the NF-UNSW-NB15 dataset. As confirmed earlier, there are no missing values or duplicate rows in this dataset.

### Applying Smote

In [ ]:
smote = SMOTE(random_state=42)
X_train2, y_train2 = smote.fit_resample(X_train2, y_train2)
y_train2.value_counts(normalize=True).round(3)


## Train & Evaluate Model

In [ ]:
models = get_models()
results2 = {}
predictions2 = {}
y_probs2 = {}

for name, model in models.items():

    model.fit(X_train2, y_train2)

    y_pred = model.predict(X_test2)
    y_prob = model.predict_proba(X_test2)[:, 1]

    accuracy = accuracy_score(y_test2, y_pred)
    f1 = f1_score(y_test2, y_pred)
    auc = roc_auc_score(y_test2, y_prob)
    precision = precision_score(y_test2, y_pred, zero_division=0)
    recall = recall_score(y_test2, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test2, y_pred, labels=[0, 1]).ravel()
    dr = tp / max((tp + fn), 1) # Detection Rate (DR)
    far = fp / max((fp + tn), 1) # False Alarm Rate (FAR).

    cv_scores = cross_val_score(model, X_train2, y_train2, cv=5, scoring='f1_macro')

    results2[name] = {
        'Accuracy': accuracy,
        'F1-Score': f1,
        'AUC': auc,
        'Precision': precision,
        'Recall': recall,
        'Detection Rate (DR)': dr,
        'False Alarm Rate (FAR)': far,
        'CV Mean F1': cv_scores.mean(),
        'CV Std F1': cv_scores.std(),
    }
    predictions2[name] = y_pred
    y_probs2[name] = y_prob # Store predicted probabilities

results_df2 = pd.DataFrame(results2).T
results_df2.index.name = 'Model'
results_df2.sort_values(by='F1-Score', ascending=False, inplace=True)
display(results_df2.round(4))


## Hyperparameter Tuning - NF-BoT-IoT

### Random Forest Hyperparameter Tuning (NF-BoT-IoT)

In [ ]:
results2, predictions2, y_probs2, best_params_rf2, best_score_rf2, tuned_rf2 = tune_and_evaluate_random_forest(
    X_train2, y_train2, X_test2, y_test2, results2, predictions2, y_probs2, 'NF-BoT-IoT', param_grid_rf
)


### Re-evaluating Random Forest with Tuned Hyperparameters - NF-BoT-IoT

In [ ]:
results_df2 = pd.DataFrame(results2).T
results_df2.index.name = 'Model'
results_df2.sort_values(by='F1-Score', ascending=False, inplace=True)
display(results_df2.round(4))


### Summary of Hyperparameter Tuning for NF-BoT-IoT

The best Random Forest hyperparameters and cross-validation F1 score are displayed dynamically in the tuning-cell output.


## Confusion Matrix Visualization

In [ ]:
fig, axes = plt.subplots(1, len(predictions2), figsize=(6 * len(predictions2), 5))

for idx, (name, y_pred) in enumerate(predictions2.items()):
    cm = confusion_matrix(y_test2, y_pred)

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=axes[idx],
        xticklabels=['Benign', 'Attack'],
        yticklabels=['Benign', 'Attack'],
    )

    axes[idx].set_title(f"{name}\nAccuracy: {results2[name]['Accuracy'] * 100:.2f}%", fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()


## Feature Importance Analysis

In [ ]:
importance_model_name = 'Random Forest'
importance_model = tuned_rf2

feature_importance2 = pd.Series(importance_model.feature_importances_, index=selected_features).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=feature_importance2.values, y=feature_importance2.index, hue=feature_importance2.index,
            palette='viridis', legend=False)
plt.title('Random Forest Feature Importance - NF-BoT-IoT')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## ROC Curve Analysis - NF-BoT-IoT

In [ ]:
from sklearn.metrics import roc_curve, auc
plt.figure(figsize=(10, 8))

for name, y_prob in y_probs2.items():
    fpr, tpr, _ = roc_curve(y_test2, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - NF-BoT-IoT', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


# DATASET -3 : NF-ToN-IoT

##importing dataset

In [ ]:
!kaggle datasets download dhoogla/nftoniot
!unzip nftoniot.zip


##Collect Data

In [ ]:
df3 = pd.read_parquet('NF-ToN-IoT.parquet')
df3.shape


In [ ]:
df3.head()



### DESCRIPTION of the Dataset:


| Feature Name | Description |
| :--- | :--- |
| **L4_SRC_PORT** | Source port number (Transport layer) |
| **L4_DST_PORT** | Destination port number (Transport layer) |
| **PROTOCOL** | IP protocol identifier (e.g., TCP=6, UDP=17, ICMP=1) |
| **TCP_FLAGS** | Cumulative TCP flags (SYN, ACK, FIN, RST, etc.) |
| **L7_PROTO** | Layer 7 protocol (Application layer protocol) |
| **IN_BYTES** | Number of incoming bytes |
| **OUT_BYTES** | Number of outgoing bytes |
| **IN_PKTS** | Number of incoming packets |
| **OUT_PKTS** | Number of outgoing packets |
| **FLOW_DURATION_MILLISECONDS** | Flow duration in milliseconds |
| **Label** | Binary label: 0=Benign, 1=Attack |
| **Attack** | Multi-class attack category (Specific attack type) |


###Sampling for tractability
The full dataset is 1.15M rows. Training three ensemble models (with SMOTE and 5-fold CV) on the full size is unnecessarily slow for what's fundamentally a feature/algorithm comparison exercise, not a production model - taking a stratified sample (preserving the original Label ratio) keeps training fast while still giving each model a large, representative sample to learn from.

In [ ]:
df3, _ = train_test_split(df3, train_size=SAMPLE_SIZE, stratify=df3['Label'], random_state=42)
df3 = df3.reset_index(drop=True)

df3.shape


##Exploratory Data Analysis (EDA)

In [ ]:
df3.shape


In [ ]:
df3.info()


In [ ]:
# check for missing values
df3.isna().sum().sum()


No missing values

In [ ]:
# check for duplicate rows
df3.duplicated().sum()


no duplicate rows

###TARGET VARIABLE ANALYSIS - BINARY CLASSIFICATION

In [ ]:
df3['Label'].value_counts()


In [ ]:
label_counts = df3['Label'].value_counts()
label_percentages = df3['Label'].value_counts(normalize=True) * 100

print(f" Label Distribution:")
print(f"   Benign  (0): {label_counts[0]:,} ({label_percentages[0]:.2f}%)")
print(f"   Attack  (1): {label_counts[1]:,} ({label_percentages[1]:.2f}%)")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
# Pie chart
colors = ['#4CAF50', '#FF5722']
ax.pie([label_counts.get(0, 0), label_counts.get(1, 0)],
            labels=['Benign', 'Attack'],
            autopct='%1.2f%%',
            colors=colors,
            explode=(0.05, 0.1),
            shadow=True,
            startangle=90)
ax.set_title('Class Distribution - NF-ToN-IoT', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


NF-ToN-IoT is benign-heavy (~82.86% benign / 17.14% attack) - **applying `SMOTE` to the `training set` only**.


### Conclusion -> `high class imbalance`


## Feature Selection & Engineering

Same as earlier

In [ ]:
selected_features = ['PROTOCOL', 'TCP_FLAGS', 'L7_PROTO', 'IN_BYTES', 'OUT_BYTES',
                     'IN_PKTS', 'OUT_PKTS', 'FLOW_DURATION_MILLISECONDS']
target_col = 'Label'

X3 = df3[selected_features].copy()
y3 = df3[target_col].astype(int)


**Verifying the selection** - checking that the 8 selected features actually separate benign from attack traffic before committing to them.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 22))
axes = axes.flatten()

for ax, feature in zip(axes, selected_features):
    benign_data = X3[y3 == 0][feature]
    attack_data = X3[y3 == 1][feature]

    bp = ax.boxplot([benign_data, attack_data], labels=['Benign', 'Attack'], medianprops={'color': "#f5f500", 'linewidth': 2}, patch_artist=True)

    colors = ['#2ecc71', "#e73c3c"] # Soft green -> Benign, Soft red ->Attack
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title(f"Distribution of {feature} - NF-ToN-IoT", fontweight='bold', fontsize=14, pad=10)
    ax.grid(True, linestyle='--', alpha=0.5, axis='y')

    if feature in ['IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS']:
        ax.set_yscale('log')

plt.tight_layout()
plt.subplots_adjust(hspace=0.4, wspace=0.3)
plt.show()


The selected features specially `OUT_BYTES, OUT_PKTS, PROTOCOL, and L7_PROTO` show clear class-separating behavior (visible spread between Benign/Attack boxes, especially in byte/packet counts on the log scale) - confirms they're worth keeping.

## Data Preprocessing

* no missing values or duplicate rows to remove (confirmed in EDA above)
* `PROTOCOL` and `L7_PROTO` are already numerically encoded in this build - no categorical encoding needed

## Split Dataset (Training & Testing) & Handle Imbalance

In [ ]:
X_train3, X_test3, y_train3, y_test3 = train_test_split(X3, y3, test_size=0.2, random_state=42, stratify=y3)

scaler3 = StandardScaler()
X_train3 = scaler3.fit_transform(X_train3)
X_test3 = scaler3.transform(X_test3)


### TARGET VARIABLE ANALYSIS

The dataset is benign-heavy (~82.86% benign / 17.14% attack) as explored earlier. So resampling of data is necessary.


### Applying Smote

In [ ]:
smote3 = SMOTE(random_state=42)
X_train3, y_train3 = smote3.fit_resample(X_train3, y_train3)
y_train3.value_counts(normalize=True).round(3)


## Train & Evaluate Model

In [ ]:
models = get_models()
results3 = {}
predictions3 = {}
y_probs3 = {}

for name, model in models.items():

    model.fit(X_train3, y_train3)

    y_pred = model.predict(X_test3)
    y_prob = model.predict_proba(X_test3)[:, 1]

    accuracy = accuracy_score(y_test3, y_pred)
    f1 = f1_score(y_test3, y_pred)
    auc = roc_auc_score(y_test3, y_prob)
    precision = precision_score(y_test3, y_pred, zero_division=0)
    recall = recall_score(y_test3, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test3, y_pred, labels=[0, 1]).ravel()
    dr = tp / max((tp + fn), 1) # Detection Rate (DR)
    far = fp / max((fp + tn), 1) # False Alarm Rate (FAR).

    cv_scores = cross_val_score(model, X_train3, y_train3, cv=5, scoring='f1_macro')

    results3[name] = {
        'Accuracy': accuracy,
        'F1-Score': f1,
        'AUC': auc,
        'Precision': precision,
        'Recall': recall,
        'Detection Rate (DR)': dr,
        'False Alarm Rate (FAR)': far,
        'CV Mean F1': cv_scores.mean(),
        'CV Std F1': cv_scores.std(),
    }
    predictions3[name] = y_pred
    y_probs3[name] = y_prob # Store predicted probabilities

results_df3 = pd.DataFrame(results3).T
results_df3.index.name = 'Model'
results_df3.sort_values(by='F1-Score', ascending=False, inplace=True)
display(results_df3.round(4))


## Hyperparameter Tuning - NF-ToN-IoT

### Random Forest Hyperparameter Tuning (NF-ToN-IoT)

In [ ]:
results3, predictions3, y_probs3, best_params_rf3, best_score_rf3, tuned_rf3 = tune_and_evaluate_random_forest(
    X_train3, y_train3, X_test3, y_test3, results3, predictions3, y_probs3, 'NF-ToN-IoT', param_grid_rf
)


### Re-evaluating Random Forest with Tuned Hyperparameters - NF-ToN-IoT

In [ ]:
results_df3 = pd.DataFrame(results3).T
results_df3.index.name = 'Model'
results_df3.sort_values(by='F1-Score', ascending=False, inplace=True)
display(results_df3.round(4))


### Summary of Hyperparameter Tuning for NF-ToN-IoT

The best Random Forest hyperparameters and cross-validation F1 score are displayed dynamically in the tuning-cell output.


## Confusion Matrix Visualization

In [ ]:
fig, axes = plt.subplots(1, len(predictions3), figsize=(6 * len(predictions3), 5))

for idx, (name, y_pred) in enumerate(predictions3.items()):
    cm = confusion_matrix(y_test3, y_pred)

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=axes[idx],
        xticklabels=['Benign', 'Attack'],
        yticklabels=['Benign', 'Attack'],
    )

    axes[idx].set_title(f"{name}\nAccuracy: {results3[name]['Accuracy'] * 100:.2f}%", fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()


## Feature Importance Analysis

In [ ]:
importance_model_name = 'Random Forest'
importance_model = tuned_rf3

feature_importance3 = pd.Series(importance_model.feature_importances_, index=selected_features).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=feature_importance3.values, y=feature_importance3.index, hue=feature_importance3.index,
            palette='viridis', legend=False)
plt.title('Random Forest Feature Importance - NF-ToN-IoT')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## ROC Curve Analysis - NF-ToN-IoT

In [ ]:
from sklearn.metrics import roc_curve, auc
plt.figure(figsize=(10, 8))

for name, y_prob in y_probs3.items():
    fpr, tpr, _ = roc_curve(y_test3, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - NF-ToN-IoT', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


# Cross-Dataset Comparison

In [ ]:
all_results = {'NF-UNSW-NB15': results_df1, 'NF-BoT-IoT': results_df2, 'NF-ToN-IoT': results_df3}
metric_cols = ['Accuracy', 'F1-Score', 'AUC', 'Detection Rate (DR)', 'False Alarm Rate (FAR)']

comparison = pd.concat(
    {dataset: df[metric_cols] for dataset, df in all_results.items()},
    names=['Dataset', 'Model']
)
comparison.round(4)


###Visualization

#### Prepare data for plotting

In [ ]:
plot_metrics = ['Accuracy', 'F1-Score', 'Detection Rate (DR)', 'False Alarm Rate (FAR)', 'AUC']
plot_data = comparison.copy()

# Metrics are already in a 0-1 range, no conversion needed.

plot_data


#### Accuracy Comparison

In [ ]:

metric = 'Accuracy'
plt.figure(figsize=(10, 6))

sns.barplot(
    data=plot_data,
    x='Model',
    y=metric,
    hue='Dataset'
)


plt.title('Accuracy Comparison', fontweight='bold')
plt.xlabel('Model')
plt.ylabel('Accuracy') # Removed (%) as values are 0-1

plt.ylim(0, 1.05) # Set y-axis limit to 0-1.05

for container in plt.gca().containers:
    plt.bar_label(container, fmt='%.3f', fontsize=8) # Changed to 3 decimal places

plt.tight_layout()
plt.show()


#### F1-Score Comparison

In [ ]:
metric = 'F1-Score'

plt.figure(figsize=(10, 6))

sns.barplot(
    data=plot_data,
    x='Model',
    y=metric,
    hue='Dataset'
)

plt.title('F1-Score Comparison', fontweight='bold') # Corrected title
plt.xlabel('Model')
plt.ylabel('F1-Score') # Corrected label

plt.ylim(0, 1.05) # Set y-axis limit to 0-1.05

for container in plt.gca().containers:
    plt.bar_label(container, fmt='%.3f', fontsize=8) # Changed to 3 decimal places

plt.tight_layout()
plt.show()


#### Detection Rate (DR) Comparison

In [ ]:
# Select the Detection Rate (DR) metric
metric = 'Detection Rate (DR)'

# Create the graph
plt.figure(figsize=(10, 6))

sns.barplot(
    data=plot_data,
    x='Model',
    y=metric,
    hue='Dataset'
)

# Add title and labels
plt.title('Detection Rate (DR) Comparison', fontweight='bold') # Corrected title
plt.xlabel('Model')
plt.ylabel('Detection Rate (DR)') # Corrected label

# Set y-axis limit
plt.ylim(0, 1.05) # Set y-axis limit to 0-1.05

# Add values on top of each bar
for container in plt.gca().containers:
    plt.bar_label(container, fmt='%.3f', fontsize=8) # Changed to 3 decimal places

plt.tight_layout()
plt.show()


#### False Alarm Rate (FAR) Comparison

In [ ]:
metric = 'False Alarm Rate (FAR)'

plt.figure(figsize=(10,6))

sns.barplot(
    data=plot_data,
    x='Model',
    y=metric,
    hue='Dataset'
)

plt.title('False Alarm Rate (FAR) Comparison', fontweight='bold') # Corrected title
plt.xlabel('Model')
plt.ylabel('False Alarm Rate (FAR)') # Corrected label

plt.ylim(0, 0.5) # Set y-axis limit to 0-1.05 for consistency

for container in plt.gca().containers:
    plt.bar_label(container, fmt='%.3f', fontsize=8)

plt.tight_layout()
plt.show()


#### AUC Comparison

In [ ]:
metric = 'AUC'

plt.figure(figsize=(10,6))

sns.barplot(
    data=plot_data,
    x='Model',
    y=metric,
    hue='Dataset'
)

plt.title('AUC Comparison', fontweight='bold')
plt.xlabel('Model')
plt.ylabel('AUC')

plt.ylim(0, 1.05)

for container in plt.gca().containers:
    plt.bar_label(container, fmt='%.3f', fontsize=8)

plt.tight_layout()
plt.show()


# Best Model Analysis

In [ ]:
for dataset_name, results_df in all_results.items():
    best_model = results_df['F1-Score'].idxmax()
    print(f"{dataset_name}: best = {best_model} (F1 = {results_df.loc[best_model, 'F1-Score']:.4f}, "
          f"DR = {results_df.loc[best_model, 'Detection Rate (DR)'] * 100:.2f}%)")
